In [0]:
-- STEP 1 --
CREATE OR REPLACE TABLE f1.silver.sessions AS

WITH latest AS (
    SELECT *
    FROM f1.bronze.openf1_sessions
    QUALIFY ROW_NUMBER() OVER (PARTITION BY session_key ORDER BY _ingested_at DESC) = 1
),

results AS (
    SELECT session_key, COUNT(*) AS result_rows
    FROM f1.bronze.openf1_session_result
    GROUP BY session_key
)

SELECT
    s.session_key,
    s.meeting_key,
    CAST(s.year AS INT) AS year,
    s.session_name,
    s.session_type,
    CAST(s.date_start AS TIMESTAMP) AS date_start,
    CAST(s.date_end AS TIMESTAMP) AS date_end,
    s.location,
    s.country_name,
    s.circuit_short_name,
    CASE
        WHEN CAST(s.date_end AS TIMESTAMP) > current_timestamp() THEN 'scheduled'
        WHEN s.session_type = 'Race' AND COALESCE(r.result_rows, 0) = 0 THEN 'cancelled'
        ELSE 'completed'
    END AS status
FROM latest s
LEFT JOIN results r ON r.session_key = s.session_key

In [0]:
SELECT year, session_type, status, COUNT(*) AS sessions
FROM f1.silver.sessions
GROUP BY year, session_type, status
ORDER BY year, session_type, status

In [0]:
-- STEP 2 --
SELECT *
FROM f1.bronze.openf1_session_result
WHERE session_key = 9693          -- Melbourne 2025, the wet race
ORDER BY position NULLS LAST
LIMIT 20 OFFSET 14

In [0]:
-- STEP 3 --
SELECT gap_to_leader, COUNT(*) AS rows
FROM f1.bronze.openf1_session_result
WHERE gap_to_leader IS NOT NULL
  AND TRY_CAST(gap_to_leader AS DOUBLE) IS NULL
GROUP BY gap_to_leader
ORDER BY rows DESC

In [0]:
SELECT COUNT(*) AS rescued_rows
FROM f1.bronze.openf1_session_result
WHERE _rescued_data IS NOT NULL

In [0]:
-- STEP 4 --
CREATE OR REPLACE TABLE f1.silver.session_results AS

WITH latest AS (
    SELECT *
    FROM f1.bronze.openf1_session_result
    QUALIFY ROW_NUMBER() OVER (PARTITION BY session_key, driver_number ORDER BY _ingested_at DESC) = 1
)

SELECT 
    session_key,
    meeting_key,
    year,
    driver_number,

    CASE
        WHEN dsq THEN 'disqualified'
        WHEN dns THEN 'did_not_start'
        WHEN dnf THEN 'retired'
        ELSE 'finished'
    END AS status,

    position,

    ROW_NUMBER() OVER (PARTITION BY session_key ORDER BY position NULLS LAST, number_of_laps DESC, driver_number)
        AS finish_order,
    
    number_of_laps AS laps_completed,
    CAST(points AS DOUBLE) AS points,
    duration AS race_time_sec,

    TRY_CAST(gap_to_leader AS DOUBLE) AS gap_to_leader_sec,
    CASE
        WHEN gap_to_leader LIKE '+%LAP'
            THEN CAST(regexp_extract(gap_to_leader, '[0-9]+', 0) AS INT)
        WHEN NOT (dnf OR dns OR dsq) THEN 0
    END AS laps_behind
FROM latest

In [0]:
-- 1. Status mix per year
SELECT year, status, COUNT(*) AS rows
FROM f1.silver.session_results
GROUP BY year, status
ORDER BY year, status;

In [0]:
-- 2. Every completed race or sprint should have exactly one winner
SELECT s.year, s.location, s.session_name,
       COUNT_IF(r.position = 1) AS winners
FROM f1.silver.sessions s
LEFT JOIN f1.silver.session_results r ON r.session_key = s.session_key
WHERE s.session_type = 'Race' AND s.status = 'completed'
GROUP BY s.year, s.location, s.session_name
HAVING winners <> 1

In [0]:
-- STEP 5 --
SELECT s.year, s.location, s.session_name, s.session_key,
       COUNT(*) AS drivers
FROM f1.silver.session_results r
JOIN f1.silver.sessions s ON s.session_key = r.session_key
GROUP BY s.year, s.location, s.session_name, s.session_key
HAVING drivers < CASE WHEN s.year = 2026 THEN 22 ELSE 20 END
ORDER BY s.year

In [0]:
SELECT d.session_key, d.driver_number, d.full_name, d.team_name
FROM f1.bronze.openf1_drivers d
LEFT ANTI JOIN f1.silver.session_results r
       ON r.session_key = d.session_key
      AND r.driver_number = d.driver_number
WHERE d.session_key IN (9165, 9069, 9488, 9971)
ORDER BY d.session_key

In [0]:
WITH target AS (
  SELECT session_key, year, date_start
  FROM f1.silver.sessions
  WHERE session_key IN (9165, 9069, 9488, 9971)
),

prev AS (
  -- The most recent completed race or sprint before each target session
  SELECT t.session_key                         AS target_key,
         MAX_BY(s.session_key, s.date_start)   AS prev_key
  FROM target t
  JOIN f1.silver.sessions s
    ON  s.year = t.year
    AND s.session_type = 'Race'
    AND s.status = 'completed'
    AND s.date_start < t.date_start
  GROUP BY t.session_key
)

SELECT p.target_key, p.prev_key, d.driver_number, d.full_name, d.team_name
FROM prev p
JOIN f1.bronze.openf1_drivers d
  ON d.session_key = p.prev_key
LEFT ANTI JOIN f1.bronze.openf1_drivers cur
  ON  cur.session_key   = p.target_key
  AND cur.driver_number = d.driver_number
ORDER BY p.target_key

In [0]:
-- STEP 6 --
-- Car numbers used by more than one driver
SELECT driver_number, COLLECT_SET(full_name) AS names
FROM f1.bronze.openf1_drivers
GROUP BY driver_number
HAVING SIZE(names) > 1

In [0]:
-- Drivers who used more than one car number, or appear under more than one name
SELECT name_acronym, COLLECT_SET(full_name) AS names, COLLECT_SET(driver_number) AS numbers
FROM f1.bronze.openf1_drivers
GROUP BY name_acronym
HAVING SIZE(names) > 1 OR SIZE(numbers) > 1

In [0]:
-- STEP 7 --
CREATE OR REPLACE TABLE f1.silver.session_drivers AS

SELECT
  -- Stable identity: "Andrea Kimi" + "Antonelli" -> andrea_kimi_antonelli
  regexp_replace(lower(concat_ws(' ', first_name, last_name)), ' ', '_') AS driver_id,
  session_key,
  meeting_key,
  year,
  driver_number,                 -- describes the car in THIS session, not the person
  first_name,
  last_name,
  name_acronym,
  team_name,
  concat('#', team_colour)       AS team_colour,   -- ready to use as a colour in dashboards
  country_code
FROM f1.bronze.openf1_drivers
QUALIFY ROW_NUMBER() OVER (PARTITION BY session_key, driver_number
                           ORDER BY _ingested_at DESC) = 1

In [0]:
-- 1. Every row has an identity
SELECT COUNT(*)                       AS rows,
       COUNT(DISTINCT driver_id)      AS drivers,
       COUNT_IF(driver_id IS NULL OR driver_id = '') AS missing_ids
FROM f1.silver.session_drivers

In [0]:
-- 2. Every result can be linked to a driver (should be 0)
SELECT COUNT(*) AS results_without_driver
FROM f1.silver.session_results r
LEFT ANTI JOIN f1.silver.session_drivers d
  ON d.session_key = r.session_key AND d.driver_number = r.driver_number

In [0]:
-- STEP 8 --
DESCRIBE f1.bronze.openf1_laps

In [0]:
SELECT
  COUNT(*)                          AS laps,
  COUNT_IF(lap_duration IS NULL)    AS no_lap_time,
  COUNT_IF(is_pit_out_lap)          AS pit_out_laps,
  MIN(lap_duration)                 AS fastest_sec,
  PERCENTILE(lap_duration, 0.5)     AS median_sec,
  MAX(lap_duration)                 AS slowest_sec
FROM f1.bronze.openf1_laps

In [0]:
-- STEP 9 --
CREATE OR REPLACE TEMP VIEW laps_clean AS

WITH latest AS (
  SELECT *
  FROM f1.bronze.openf1_laps
  QUALIFY ROW_NUMBER() OVER (PARTITION BY session_key, driver_number, lap_number
                             ORDER BY _ingested_at DESC) = 1
),

typed AS (
  SELECT
    session_key, meeting_key, year, driver_number, lap_number,
    CAST(date_start AS TIMESTAMP)  AS lap_start,
    lap_duration                   AS lap_time_sec,
    duration_sector_1              AS sector_1_sec,
    duration_sector_2              AS sector_2_sec,
    duration_sector_3              AS sector_3_sec,
    i1_speed                       AS speed_i1_kph,
    i2_speed                       AS speed_i2_kph,
    st_speed                       AS speed_trap_kph,
    COALESCE(is_pit_out_lap, false) AS is_pit_out_lap,
    -- The median lap of the session = normal racing pace
    PERCENTILE(lap_duration, 0.5) OVER (PARTITION BY session_key) AS session_median_sec
  FROM latest
)

SELECT
  * EXCEPT (session_median_sec),

  -- Representative of race pace: timed, not lap 1, not an out-lap,
  -- and not much slower than normal (safety car, red flag, incident)
  (lap_time_sec IS NOT NULL
   AND lap_number > 1
   AND NOT is_pit_out_lap
   AND lap_time_sec <= 1.10 * session_median_sec) AS is_clean_lap,

  -- Structural problems that make a row unusable
  CASE
    WHEN session_key IS NULL OR driver_number IS NULL OR lap_number IS NULL THEN 'missing key'
    WHEN lap_time_sec <= 0 THEN 'non-positive lap time'
  END AS _issue
FROM typed

In [0]:
-- Create the empty table the first time (LIMIT 0 copies only the columns)
CREATE TABLE IF NOT EXISTS f1.silver.laps AS
SELECT * EXCEPT (_issue) FROM laps_clean LIMIT 0;

-- Upsert: new laps are inserted, existing laps are updated
MERGE INTO f1.silver.laps AS t
USING (SELECT * EXCEPT (_issue) FROM laps_clean WHERE _issue IS NULL) AS s
ON  t.session_key   = s.session_key
AND t.driver_number = s.driver_number
AND t.lap_number    = s.lap_number
WHEN MATCHED     THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- Broken rows go to quarantine instead of disappearing silently
CREATE OR REPLACE TABLE f1.silver.quarantine_laps AS
SELECT * FROM laps_clean WHERE _issue IS NOT NULL;

In [0]:
SELECT
  (SELECT COUNT(*) FROM f1.silver.laps)            AS laps,
  (SELECT COUNT_IF(is_clean_lap) FROM f1.silver.laps) AS clean_laps,
  (SELECT COUNT(*) FROM f1.silver.quarantine_laps) AS quarantined

In [0]:
-- Where are the untimed laps?
SELECT lap_number = 1 AS is_lap_1, COUNT(*) AS untimed_laps
FROM f1.silver.laps
WHERE lap_time_sec IS NULL
GROUP BY 1

In [0]:
-- STEP 10 --
DESCRIBE HISTORY f1.silver.laps

In [0]:
DESCRIBE f1.bronze.openf1_stints

In [0]:
DESCRIBE f1.bronze.openf1_pit

In [0]:
-- Which tyre compounds appear, and how often?
SELECT compound, COUNT(*) AS stints
FROM f1.bronze.openf1_stints
GROUP BY compound
ORDER BY stints DESC

In [0]:
-- STEP 11 --
CREATE OR REPLACE TABLE f1.silver.stints AS

SELECT
  session_key,
  meeting_key,
  year,
  driver_number,
  stint_number,
  CASE
    WHEN compound IS NULL OR compound = 'UNKNOWN' THEN 'UNKNOWN'
    ELSE upper(compound)
  END                                         AS compound,
  CASE
    WHEN compound IN ('SOFT', 'MEDIUM', 'HARD')  THEN 'dry'
    WHEN compound IN ('INTERMEDIATE', 'WET')     THEN 'wet'
    ELSE 'unknown'
  END                                         AS compound_type,
  lap_start,
  lap_end,
  lap_end - lap_start + 1                     AS stint_laps,
  tyre_age_at_start                           AS tyre_age_start    -- >0 means used tyres
FROM f1.bronze.openf1_stints
QUALIFY ROW_NUMBER() OVER (PARTITION BY session_key, driver_number, stint_number
                           ORDER BY _ingested_at DESC) = 1

In [0]:
SELECT
  COUNT(*)                            AS stints,
  COUNT_IF(lap_end IS NULL)           AS missing_end,
  COUNT_IF(lap_end < lap_start)       AS end_before_start,
  MAX(stint_laps)                     AS longest_stint
FROM f1.silver.stints

In [0]:
SELECT year,
       COUNT(*)              AS stops,
       COUNT(pit_duration)   AS has_pit_duration,
       COUNT(lane_duration)  AS has_lane_duration,
       COUNT(stop_duration)  AS has_stop_duration
FROM f1.bronze.openf1_pit
GROUP BY year
ORDER BY year

In [0]:
CREATE OR REPLACE TEMP VIEW stints_clean AS

WITH latest AS (
  SELECT *
  FROM f1.bronze.openf1_stints
  QUALIFY ROW_NUMBER() OVER (PARTITION BY session_key, driver_number, stint_number
                             ORDER BY _ingested_at DESC) = 1
),

filled AS (
  SELECT
    st.*,
    st.lap_end IS NULL AS lap_end_inferred,
    -- 1) Fill a missing end, 2) never end on or after the next stint's start lap
    LEAST(
      COALESCE(
        st.lap_end,
        LEAD(st.lap_start) OVER w - 1,
        r.laps_completed
      ),
      LEAD(st.lap_start) OVER w - 1
    ) AS lap_end_filled,
    st.lap_end > LEAD(st.lap_start) OVER w - 1 AS lap_end_trimmed
  FROM latest st
  LEFT JOIN f1.silver.session_results r
         ON r.session_key = st.session_key
        AND r.driver_number = st.driver_number
  WINDOW w AS (PARTITION BY st.session_key, st.driver_number ORDER BY st.stint_number)
),

flagged AS (
  SELECT *,
    (lap_start IS NULL AND lap_end_filled = 0)
      OR (lap_end_filled = lap_start - 1)          AS is_zero_lap_stint
  FROM filled
)

SELECT
  session_key, meeting_key, year, driver_number, stint_number,
  CASE WHEN compound IS NULL OR compound = 'UNKNOWN' THEN 'UNKNOWN'
       ELSE upper(compound) END                       AS compound,
  CASE WHEN compound IN ('SOFT', 'MEDIUM', 'HARD')    THEN 'dry'
       WHEN compound IN ('INTERMEDIATE', 'WET')       THEN 'wet'
       ELSE 'unknown' END                             AS compound_type,
  lap_start,
  lap_end_filled                                      AS lap_end,
  lap_end_inferred,
  COALESCE(lap_end_trimmed, false)                    AS lap_end_trimmed,
  is_zero_lap_stint,
  CASE WHEN is_zero_lap_stint THEN 0
       ELSE lap_end_filled - lap_start + 1 END        AS stint_laps,
  tyre_age_at_start                                   AS tyre_age_start,
  CASE
    WHEN is_zero_lap_stint                        THEN NULL
    WHEN lap_start IS NULL                        THEN 'missing lap_start'
    WHEN lap_end_filled < lap_start               THEN 'ends before it starts'
  END                                                 AS _issue
FROM flagged

In [0]:
CREATE OR REPLACE TABLE f1.silver.stints AS
SELECT * EXCEPT (_issue) FROM stints_clean WHERE _issue IS NULL;

CREATE OR REPLACE TABLE f1.silver.quarantine_stints AS
SELECT * FROM stints_clean WHERE _issue IS NOT NULL;

CREATE OR REPLACE TABLE f1.silver.pit_stops AS
SELECT
  session_key, meeting_key, year, driver_number, lap_number,
  CAST(date AS TIMESTAMP)                 AS pit_entry_time,
  COALESCE(lane_duration, pit_duration)   AS lane_time_sec,       -- pit entry to exit
  stop_duration                           AS stationary_time_sec  -- null = not measured
FROM f1.bronze.openf1_pit
QUALIFY ROW_NUMBER() OVER (PARTITION BY session_key, driver_number, lap_number
                           ORDER BY _ingested_at DESC) = 1;

In [0]:
-- 1. Is pit_duration really the same as lane_duration? (should be 0)
SELECT COUNT_IF(pit_duration <> lane_duration) AS rows_that_differ
FROM f1.bronze.openf1_pit

In [0]:
-- 2. Stints after the fix
SELECT COUNT(*)                     AS stints,
       COUNT_IF(lap_end IS NULL)    AS still_missing_end,
       COUNT_IF(lap_end_inferred)   AS ends_filled_in
FROM f1.silver.stints

In [0]:
-- 3. What did the quarantine catch?
SELECT * FROM f1.silver.quarantine_stints
ORDER BY session_key, driver_number, stint_number

In [0]:
SELECT
  (SELECT COUNT(*)                    FROM f1.silver.stints)            AS stints,
  (SELECT COUNT_IF(is_zero_lap_stint) FROM f1.silver.stints)            AS zero_lap_stints,
  (SELECT COUNT(*)                    FROM f1.silver.quarantine_stints) AS quarantined

In [0]:
SELECT * FROM f1.silver.quarantine_stints

In [0]:
-- STEP 13 --
CREATE OR REPLACE TABLE f1.silver.starting_grid AS

WITH grid AS (
  SELECT *
  FROM f1.bronze.openf1_starting_grid
  QUALIFY ROW_NUMBER() OVER (PARTITION BY session_key, driver_number
                             ORDER BY _ingested_at DESC) = 1
)

SELECT
  race.session_key            AS session_key,             -- the race or sprint this grid is for
  g.session_key               AS qualifying_session_key,  -- where OpenF1 stored it
  g.meeting_key,
  q.year,
  g.driver_number,
  g.position                  AS grid_position,
  g.lap_duration              AS qualifying_lap_sec
FROM grid g
JOIN f1.silver.sessions q
  ON q.session_key = g.session_key
LEFT JOIN f1.silver.sessions race
  ON  race.meeting_key  = g.meeting_key
  AND race.session_name = CASE
        WHEN q.session_name = 'Qualifying'                              THEN 'Race'
        WHEN q.session_name IN ('Sprint Qualifying', 'Sprint Shootout') THEN 'Sprint'
      END

In [0]:
-- 1. All grid rows kept, all linked to a race (expect 2,240 and 0)
SELECT COUNT(*)                       AS rows,
       COUNT_IF(session_key IS NULL)  AS not_linked
FROM f1.silver.starting_grid

In [0]:
-- 2. Completed races and sprints with no grid (expect only 2025 São Paulo Sprint)
SELECT s.year, s.location, s.session_name
FROM f1.silver.sessions s
LEFT ANTI JOIN f1.silver.starting_grid g
  ON g.session_key = s.session_key
WHERE s.session_type = 'Race'
  AND s.status = 'completed'

In [0]:
-- STEP 14 --
CREATE OR REPLACE TABLE f1.silver.meetings AS

WITH latest AS (
  SELECT *
  FROM f1.bronze.openf1_meetings
  QUALIFY ROW_NUMBER() OVER (PARTITION BY meeting_key ORDER BY _ingested_at DESC) = 1
),

race_status AS (
  -- What happened to each weekend's main race (from silver.sessions)
  SELECT meeting_key, MAX(CASE WHEN session_name = 'Race' THEN status END) AS race_status
  FROM f1.silver.sessions
  GROUP BY meeting_key
)

SELECT
  m.meeting_key,
  CAST(m.year AS INT)                  AS year,
  m.meeting_name,
  m.meeting_official_name,
  m.location,
  m.country_name,
  m.circuit_short_name,
  CAST(m.date_start AS TIMESTAMP)      AS date_start,
  m.meeting_name ILIKE '%testing%'     AS is_testing,
  COALESCE(r.race_status, 'no race')   AS race_status   -- completed / cancelled / scheduled
FROM latest m
LEFT JOIN race_status r ON r.meeting_key = m.meeting_key

In [0]:
CREATE OR REPLACE TABLE f1.silver.weather AS

WITH latest AS (
  SELECT *
  FROM f1.bronze.openf1_weather
  QUALIFY ROW_NUMBER() OVER (PARTITION BY session_key, date ORDER BY _ingested_at DESC) = 1
)

SELECT
  session_key, meeting_key, year,
  CAST(date AS TIMESTAMP)   AS measured_at,
  air_temperature           AS air_temp_c,
  NULLIF(track_temperature, 0)   AS track_temp_c,     -- 0 = sensor gap, not a real reading
  humidity                  AS humidity_pct,
  pressure                  AS pressure_mbar,
  rainfall > 0              AS is_raining,
  wind_speed                AS wind_speed_ms,
  wind_direction            AS wind_direction_deg
FROM latest

In [0]:
CREATE OR REPLACE TABLE f1.silver.race_control AS

WITH latest AS (
  SELECT *
  FROM f1.bronze.openf1_race_control
  QUALIFY ROW_NUMBER() OVER (PARTITION BY session_key, date, message
                             ORDER BY _ingested_at DESC) = 1
)

SELECT
  * EXCEPT (date, _rescued_data, _source_file, _ingested_at),
  CAST(date AS TIMESTAMP) AS message_time
FROM latest

In [0]:
-- Meetings per year: expect 1 testing meeting per year (maybe 2 in 2026),
-- 1 cancelled in 2023, 2 cancelled and ~6 scheduled in 2026
SELECT year, is_testing, race_status, COUNT(*) AS meetings
FROM f1.silver.meetings
GROUP BY ALL
ORDER BY ALL

In [0]:
-- Weather sanity: air 10–40 °C and track 15–60 °C are normal
SELECT MIN(air_temp_c)   AS min_air,   MAX(air_temp_c)   AS max_air,
       MIN(track_temp_c) AS min_track, MAX(track_temp_c) AS max_track,
       ROUND(AVG(CAST(is_raining AS INT)) * 100, 1) AS pct_readings_raining
FROM f1.silver.weather

In [0]:
-- STEP 16 --
SELECT session_key, COUNT(*) AS zero_readings
FROM f1.silver.weather
WHERE track_temp_c = 0
GROUP BY session_key
ORDER BY zero_readings DESC